<a href="https://colab.research.google.com/github/ashuyad/learning-transformers/blob/main/notebooks/semantic_search_with_FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset

In [2]:
issues_dataset = load_dataset("ashuyad1234/github-issues", split='train')

In [3]:
issues_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment', 'is_pull_request'],
    num_rows: 4000
})

In [4]:
issues_dataset[2]

{'url': 'https://api.github.com/repos/huggingface/datasets/issues/9007',
 'repository_url': 'https://api.github.com/repos/huggingface/datasets',
 'labels_url': 'https://api.github.com/repos/huggingface/datasets/issues/9007/labels{/name}',
 'comments_url': 'https://api.github.com/repos/huggingface/datasets/issues/9007/comments',
 'events_url': 'https://api.github.com/repos/huggingface/datasets/issues/9007/events',
 'html_url': 'https://github.com/huggingface/datasets/pull/9007',
 'id': 6342406378,
 'node_id': 'PR_kwDODunzps8AAAABOi-Sug',
 'number': 9007,
 'title': 'Document loading splits from files inside one ZIP archive',
 'user': {'login': 'behroozazarkhalili',
  'id': 80390531,
  'node_id': 'MDQ6VXNlcjgwMzkwNTMx',
  'avatar_url': 'https://avatars.githubusercontent.com/u/80390531?v=4',
  'gravatar_id': '',
  'url': 'https://api.github.com/users/behroozazarkhalili',
  'html_url': 'https://github.com/behroozazarkhalili',
  'followers_url': 'https://api.github.com/users/behroozazarkhali

In [5]:
issues_dataset = issues_dataset.filter(
    lambda x: (x["is_pull_request"] == False and len(x["comments"]) > 0)
)
issues_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment', 'is_pull_request'],
    num_rows: 1400
})

In [6]:
columns = issues_dataset.column_names
columns_to_keep = ["title", "body", "html_url", "comments"]
columns_to_remove = set(columns).difference(columns_to_keep)
issues_dataset = issues_dataset.remove_columns(columns_to_remove)
issues_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 1400
})

In [7]:
issues_dataset.set_format('pandas')
df = issues_dataset[:]

In [8]:
df['comments'][1].tolist()

['i can take this. plan is to add the dead worker check from #5976 around the ordered `imap` in the csv/json/sql writers and in `parallel_map`, so they raise instead of hanging, plus a test with a worker that calls `os._exit`',
 "Thanks for offering! I've already implemented this (the #5976 check around the ordered `imap` in the three writers and in `parallel_map`, with tests where a worker exits mid-task) and will open the PR later today. A review would be very welcome."]

In [9]:
comments_df = df.explode('comments', ignore_index=True)
comments_df.head(4)

,html_url,title,comments,body
0,https://github.com/huggingface/datasets/issues...,class_encode_column and align_labels_with_mapp...,#self-assign,### Describe the bug\n\n`Dataset.class_encode_...
1,https://github.com/huggingface/datasets/issues...,"With `num_proc>1`, `to_sql`/`to_csv`/`to_json`...",i can take this. plan is to add the dead worke...,"### Describe the bug\n\nWith `num_proc > 1`, t..."
2,https://github.com/huggingface/datasets/issues...,"With `num_proc>1`, `to_sql`/`to_csv`/`to_json`...",Thanks for offering! I've already implemented ...,"### Describe the bug\n\nWith `num_proc > 1`, t..."
3,https://github.com/huggingface/datasets/issues...,Signature mismatch in FilteredExamplesIterable...,Closing as not a user-facing bug (internal cal...,### Describe the bug\n\nIn `datasets.iterable_...


In [10]:
from datasets import Dataset
comments_dataset = Dataset.from_pandas(comments_df)
comments_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 5264
})

In [11]:
comments_dataset = comments_dataset.map(
    lambda x: { "comment_length" : len(x["comments"].split())}
)

Map:   0%|          | 0/5264 [00:00<?, ? examples/s]

In [12]:
comments_dataset = comments_dataset.filter(
    lambda x: x["comment_length"] > 15
)
comments_dataset

Filter:   0%|          | 0/5264 [00:00<?, ? examples/s]

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length'],
    num_rows: 3839
})

In [13]:
def concatenate_text(example):
  title = example["title"] if example["title"] is not None else ""
  body = example["body"] if example["body"] is not None else ""
  comments = example["comments"] if example["comments"] is not None else ""
  return {
      "text": title + "\n" + body + "\n" + comments
  }

comments_dataset = comments_dataset.map(concatenate_text)

Map:   0%|          | 0/3839 [00:00<?, ? examples/s]

In [14]:
comments_dataset
comments_dataset[0]

{'html_url': 'https://github.com/huggingface/datasets/issues/8993',
 'title': 'With `num_proc>1`, `to_sql`/`to_csv`/`to_json` and `load_dataset` downloads never return if a worker process dies',
 'comments': 'i can take this. plan is to add the dead worker check from #5976 around the ordered `imap` in the csv/json/sql writers and in `parallel_map`, so they raise instead of hanging, plus a test with a worker that calls `os._exit`',
 'body': '### Describe the bug\n\nWith `num_proc > 1`, two code paths send their tasks to a stdlib `multiprocessing.Pool` and wait on it without the dead-worker check that #5976 added to `iflatmap_unordered`:\n\n- the writers behind `Dataset.to_sql`, `to_csv` and `to_json`, which iterate over `pool.imap(...)` (`io/sql.py`, `io/csv.py`, `io/json.py`);\n- `datasets.parallel.parallel_map`, which `map_nested` uses and which calls `pool.map(...)` (`parallel/parallel.py`). It runs the download and extract step of `load_dataset(..., num_proc=N)` and `Dataset.from_cs

In [15]:
from transformers import AutoTokenizer, AutoModel

checkpoint = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [16]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model moved to: {device}")

Model moved to: cuda


In [17]:
def cls_pooling(model_output):
  return model_output.last_hidden_state[:, 0]

In [18]:
def get_embeddings(text_list):
  encoded_input = tokenizer(text_list, padding=True, truncation=True, return_tensors='pt')

  encoded_input = {k: v.to(device) for k,v in encoded_input.items()}
  model_output = model(**encoded_input)
  return cls_pooling(model_output)

In [19]:
embedding = get_embeddings(comments_dataset['text'][0])
embedding.shape

torch.Size([1, 768])

In [20]:
embedding_dataset = comments_dataset.map(
    lambda x: {"embeddings" : get_embeddings(x["text"]).detach().cpu().numpy()[0]}
)

Map:   0%|          | 0/3839 [00:00<?, ? examples/s]

In [21]:
!pip install faiss-cpu
embedding_dataset.add_faiss_index(column='embeddings')

  0%|          | 0/4 [00:00<?, ?it/s]

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length', 'text', 'embeddings'],
    num_rows: 3839
})

In [29]:
question = "How to create dataset viewer offline ?"
question_embedding = get_embeddings([question]).cpu().detach().numpy()
question_embedding.shape

(1, 768)

In [30]:
score, samples = embedding_dataset.get_nearest_examples("embeddings", question_embedding, k=5)

In [31]:
import pandas as pd

sample_df = pd.DataFrame.from_dict(samples)
sample_df["score"] = score
sample_df.sort_values("score", ascending=False, inplace=True)

In [32]:
for _, row in sample_df.iterrows():
    print(f"COMMENT: {row.comments}")
    print(f"SCORE: {row.score}")
    print(f"TITLE: {row.title}")
    print(f"URL: {row.html_url}")
    print("=" * 50)
    print()

COMMENT: Due to license restriction, this dataset needs manual downloading of the original data.

This information is in the dataset card: https://huggingface.co/datasets/timit_asr
> The dataset needs to be downloaded manually from https://catalog.ldc.upenn.edu/LDC93S1
SCORE: 32.05952453613281
TITLE: Dataset Viewer issue for timit_asr
URL: https://github.com/huggingface/datasets/issues/4898

COMMENT: Hi @ssuwelack I tried loading a HF dataset with your viewer but got this error https://github.com/Renumics/spotlight/issues/461 hope the team can help me on this. Thanks!
SCORE: 22.710670471191406
TITLE: Offline dataset viewer
URL: https://github.com/huggingface/datasets/issues/6139

COMMENT: Hi, we are building an offline dataset viewer: https://github.com/Renumics/spotlight
It supports many HF datasets, but currently you have to use it via Pandas:
df=ds.to_pandas()
spotlight.show(df)

Would love to hear from you if that works for your use case. If not, feel free to open an issue on the r